In [4]:
import glob
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

import numpy as np

def calculate_djf_mean(ds):
    ds_seasons = ds.resample(time='QE-FEB').mean()
    ds_djf = ds_seasons.sel(time=(ds_seasons['time.season'] == 'DJF'))
    ds_djf['time'] = pd.date_range(str(ds_djf.time[0].values).split('-')[0], str(int(str(ds_djf.time[-1].values).split('-')[0]) +1), freq='YS-FEB',unit='s')
    return ds_djf


def anomaly_calculation(model, clim):
    ano_list = []
    for i in range(1,13):
        ano_list.append(model.sel(time=model.time[model.time.dt.month.isin(i)]) - clim.sel(month=i))
    return xr.concat(ano_list, dim='time').sortby('time')


def load_climatology(path):
    clim = xr.open_dataset(glob.glob(path)[0])
    clim['time'] = [1,2,3,4,5,6,7,8,9,10,11,12]
    clim = clim.rename({'time': 'month'})
    return clim

def load_data(path):
    ens = []
    ds = []
    for p in glob.glob(path):
        ens.append(p.split('/')[-1].split('_')[4])
        ds.append(xr.open_dataset(p))
    data = xr.concat(ds, dim='ens')
    data['ens'] = ens

    return data

def pressure_weighting(data, plevs=slice(7000,2000)):
    # here I select the pressure level intervall 
    # for which I want to caluculate the layer weights
    # (in this case plevs = 70 - 20 hPa)
    data = data.sel(plev=plevs)
    # Then natural logarithm of this pressure values is caluculated
    log_pressure = np.log(data.plev.values)  
    # the difference between this logaritmic pressure values is made
    log_layer_thickness = np.abs(np.diff(log_pressure))
    # then i devide these differences by the total thickness of the layer
    # = sum of the differences
    log_layer_thickness = log_layer_thickness / sum(log_layer_thickness)

    # this part just creates an array to make it compatible with the data 
    # #(no manipulation of the results done in this step)
    log_layer_thickness = xr.DataArray(
        log_layer_thickness,
        coords={"plev": data.plev.values[:-1]},  # Assign to the lower pressure level of each layer
        dims=["plev"]
    )

    # finally I multiply the data with the "normalized" thickness of the pressure layer
    data = data * log_layer_thickness

    return data.sum(dim='plev')

In [5]:
path = "/sto4/data/Intermediate/VolMIP/pinatubo-full/"
path_piControl = "/sto4/data/Intermediate/VolMIP/piControl/"
models = ['CanESM5', 'GISS-E2-1-G','MPI-ESM1-2-LR', 'MIROC-ES2L', 'UKESM1-0-LL']

enso_nao_flags = pd.read_csv('/home/kperny/VolMIP_stratosphere/tab/nao_enso_flags.csv')
enso_nao_flags = enso_nao_flags.set_index(['model', 'realization'])


In [6]:
# calculation for piControl
for model in models:
    # open climatology
    ua_clim_pi = load_climatology(path_piControl + model + '/ua_clim_AmonZ*')
    ta_clim_pi = load_climatology(path_piControl + model + '/ta_clim_AmonZ*')
    
    # open temperature and wind data
    ta_tmp = xr.open_dataset(glob.glob(path_piControl + model + '/ta_AmonZ*.nc')[0], decode_times=time_coder)
    ta_ano = anomaly_calculation(ta_tmp, ta_clim_pi)
    ta_piControl = calculate_djf_mean(ta_ano)

    ua_tmp = xr.open_dataset(glob.glob(path_piControl + model + '/ua_AmonZ*.nc')[0], decode_times=time_coder)
    ua_ano = anomaly_calculation(ua_tmp, ua_clim_pi)
    ua_piControl = calculate_djf_mean(ua_ano)

    # agregate tropical (-20,20) and polar (70,90) temperature between 70-20hPa and 10 hPa wind data (60,70)
    enso_nao_flags.loc[model,'deltaT_tropics'] = pressure_weighting(ta_piControl.sel(time = enso_nao_flags.loc[model]['year'].values, method='nearest').ta.sel(lat=slice(-20,20)).mean('lat')).squeeze().values
    enso_nao_flags.loc[model,'deltaT_polar'] = pressure_weighting(ta_piControl.sel(time = enso_nao_flags.loc[model]['year'].values, method='nearest').ta.sel(lat=slice(70,90)).mean('lat')).squeeze().values
    enso_nao_flags.loc[model,'deltaU'] = ua_piControl.sel(time = enso_nao_flags.loc[model]['year'].values, method='nearest').ua.sel(lat=slice(60,70), plev=1000).mean('lat').squeeze().values
enso_nao_flags.reset_index().to_csv('/home/kperny/VolMIP_stratosphere/tab/ta_ua_piControl.csv', index=False)

/tmp/ipykernel_2921750/1070168184.py:27: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  clim = xr.open_dataset(glob.glob(path)[0])
/tmp/ipykernel_2921750/1070168184.py:27: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  clim = xr.open_dataset(glob.glob(path)[0])
/tmp/ipykernel_2921750/1070168184.py:22: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  ano_list.ap